# 4.5 — SmartReplay (paid)

> Run `4.0_cl_setup.ipynb` first.

**Mechanism:** keep 15 exemplars per class and mix them into every training batch.

**Memory:** ~960 KB/class at boundary B (`15 x 16 x 2048` floats), ~15 KB at boundary A.

## Exemplar selection

`strategy="hard"` keeps the samples **farthest** from each class prototype — boundary
cases carry more information than the centroid. `diverse` (greedy coreset) and `random`
are also available; Section 5 ablates all three.

## The comparison that matters

Not "does it beat Naive" — it will. The real question is **does it beat WeightAlign and
LwF (4.2, 4.4), which are free?** If not, the memory is not earning its cost.

## Setup

In [ ]:
# ── ULAL bootstrap — self-contained, no Drive dependency ────────────────────
# One shallow git clone instead of hundreds of API calls. subprocess takes a
# LIST, never a shell string, so the '&' in the branch name is safe.
import os, sys, subprocess
from getpass import getpass

REPO, BRANCH, ROOT = "silvergjeka22/Unified-Lifelong-Action-Learning", "GAN&ActiveLearning", "/content/ulal"
token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token (hidden): ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(f"{ROOT}/.git"):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    f"https://{token}@github.com/{REPO}.git", ROOT], check=True)

sys.path.insert(0, ROOT)
print(subprocess.run(["git", "-C", ROOT, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
# ── Drive + dataset + paths ─────────────────────────────────────────────────
# Exports ULAL_* env vars that config.py reads. Nothing is written to disk and
# no kernel restart is needed.
os.environ["KAGGLE_USERNAME"] = "YOUR_KAGGLE_USER"
os.environ["KAGGLE_KEY"]      = "YOUR_KAGGLE_KEY"

from src.bootstrap import setup
setup(drive=True, dataset=True)

In [ ]:
%run /content/ulal/src/imports.py

In [ ]:
# ── Global label space (src/data/cache.py) ──────────────────────────────────
TASK_CLASSES = [cfg.SELECTED_CLASSES, cfg.TASK_1, cfg.TASK_2, cfg.TASK_4]
TASK_ROOTS   = [cfg.BASE_ROOT, cfg.TASK1_ROOT, cfg.TASK2_ROOT, cfg.TASK4_ROOT]
TASK_NAMES   = ["Base", "Task1", "Task2", "Task4"]
CL_TASKS     = [1, 2]     # Sections 4-5 stream over these
FEWSHOT_TASK = 3          # Section 7 probe only

global_classes, global_c2i, NUM_AFTER = build_label_space(TASK_CLASSES)
describe_label_space(TASK_CLASSES, TASK_NAMES, NUM_AFTER, task_3=cfg.TASK_3)

In [ ]:
# ── Load cached frame features (src/data/cache.py) ──────────────────────────
CACHE, LSTM_STATE, META = load_feature_cache(cfg.FEAT_CACHE)

In [ ]:
# ── Shared protocol (written by 4.0_cl_setup) ───────────────────────────────
# Every arm loads the SAME base head and the SAME hyperparameters, so lr/epochs/
# schedule are identical by construction even though the arms run in separate
# notebooks. The mechanism flags are the only difference.
P = json.load(open(f"{cfg.RESULTS_DIR}/stage1_protocol.json"))

FREEZE_LSTM = P["freeze_lstm"]
CEILING     = P["ceiling"]
CL_EPOCHS   = P["epochs"]
CL_LR       = P["lr"]
CL_WD       = P["wd"]
CL_BATCH    = P["batch_size"]
CL_LS       = P["label_smoothing"]

base_head = make_temporal_head(NUM_AFTER[0], freeze_lstm=FREEZE_LSTM, device=device)
base_head.load_state_dict(torch.load(f"{cfg.CKPT_DIR}/head_base.pt", map_location=device))
base_head.eval()

STREAM = dict(
    base_head=base_head, cache=CACHE, cl_tasks=CL_TASKS, task_classes=TASK_CLASSES,
    task_names=TASK_NAMES, num_after=NUM_AFTER, global_c2i=global_c2i,
    base_classes=cfg.SELECTED_CLASSES, device=device,
    epochs=CL_EPOCHS, lr=CL_LR, wd=CL_WD, batch_size=CL_BATCH,
    label_smoothing=CL_LS,
)

print(f"Protocol : {CL_EPOCHS} epochs, lr={CL_LR}, batch={CL_BATCH}, no best-val selection")
print(f"Boundary : LSTM {'FROZEN' if FREEZE_LSTM else 'TRAINABLE'}")
print(f"Ceiling  : {CEILING:.2%}" if CEILING else "Ceiling  : (none — run Section 3)")
print(f"Base head: {base_head.trainable_params():,} trainable, "
      f"base acc {eval_tasks_upto(base_head, CACHE, 0, device)[0]:.2%}")

## Run the arm

In [ ]:
REPLAY_PER_CLASS = 15
REPLAY_STRATEGY  = "hard"     # 'hard' | 'diverse' | 'random'

res = run_cl_stream(arm="replay", replay_per_class=REPLAY_PER_CLASS,
                    replay_strategy=REPLAY_STRATEGY, **STREAM)

R, m = save_arm_result(
    f"{cfg.RESULTS_DIR}/stage1_arm_smart_replay.json",
    "SmartReplay", res["rows"], TASK_NAMES, res["bytes_per_class"],
    config={"mechanism": "replay", "per_class": REPLAY_PER_CLASS,
            "strategy": REPLAY_STRATEGY}, ceiling=CEILING,
)
cl_report(R, "SmartReplay", [TASK_NAMES[i] for i in [0] + CL_TASKS])
torch.save(res["head"].state_dict(), f"{cfg.CKPT_DIR}/head_smart_replay.pt")

---

**A likely observation:** Task1 may collapse after Task2 even *with* replay, while Base
holds. Not a bug — Task2 gets ~100 samples/class while replayed Task1 gets 15, a ~6.7x
vote imbalance, and the competing classes sit in the same feature region. Section 5 tests
exactly that by raising the budget.

**Next:** 4.6 adds LwF on top of this replay.